# Data Strategy (MIDA II): Sampling, Assignment, and Ethics

**Topic 07 · 2 lectures**

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/honr_464_logo.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>HONR 46400 — Evidence-Driven Research</center>
# <center>Professor: Davi Moreira</center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/notebooks/student/nb07_data_strategy_student.ipynb)

---

## 🧭 Inquiry & Claim Boundary

**Inquiry emphasis:** generalization and the causal kind. This topic builds the
Data strategy of MIDA, the moment your design stops being about ideas and
starts being about who gets into the evidence. Two compass crossings get bought
here. **Sampling** (who you get to see) buys the reach from your sample to a
population. **Assignment** (what the people you see are exposed to) buys the
crossing from a descriptive question to a causal one. Confuse the two, and
every claim downstream inherits the confusion.

| | |
|---|---|
| **A claim this topic PERMITS** | "My data strategy samples [units] by [procedure], so the results speak to [population] with stated uncertainty." |
| **A claim this topic does NOT permit** | Convenience-sample results narrated as population facts ("voters in my file" ≠ "voters in Los Angeles"), or a design whose data cannot exist before the semester ends. |

**Where this sits in the course:** Phase 3, data and answer strategies. It
opens MIDA's **D** (Data strategy) after nb04 built **M** and **I**, and it
develops milestone M05, your Data Strategy + Feasibility & Ethics checkpoint.
You complete and submit M05 during the async module at the end of that week,
as a 90-second recorded statement posted to the discussion board.

*Provenance: RDSS ch. 8 'Crafting a data strategy' §8.1 + declaration_2.1 concept | sampling & assignment within a data strategy | simple-random-sampling and complete-random-assignment simulations + honors-scale feasibility/ethics rubric | translated (R→Python) & extended*

## Learning Objectives

By the end of this notebook, you will be able to:

1. Distinguish **population**, **sampling frame**, and **sample**, and say
   which one any given claim silently invokes.
2. Show, by simulation, that random samples cluster around the truth while a
   convenience sample can miss it badly, and explain why.
3. Separate the two randomizations, random sampling (who you see) and random
   assignment (what they get), and say what each one buys.
4. Run **complete random assignment** by seeded shuffle and check covariate
   **balance** against a coin flip and against no randomization at all.
5. Triage a design for feasibility (data in time, skills, cost, consent) at
   the honors scale.
6. Walk the ethics decision tree for research involving people and write the
   feasibility and ethics paragraphs of your own M05.

---

In [ ]:
# Setup — run this cell first.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (9, 5)

SEED = 464  # course number — keeps every simulation reproducible
rng = np.random.default_rng(SEED)

# Data loads: GitHub raw URL first (works in Colab), local repo path as fallback.
DATA_URL = ("https://raw.githubusercontent.com/davi-moreira/"
            "2026F_evidence_driven_research_purdue_HONR464/main/notebooks/data/")

def load_course_data(filename):
    """Load a course dataset from GitHub, falling back to the local repo copy."""
    try:
        return pd.read_csv(DATA_URL + filename)
    except Exception:
        from pathlib import Path
        local = Path("notebooks/data") / filename
        if not local.exists():
            local = Path("../data") / filename
        return pd.read_csv(local)

print("✓ Setup complete — seed", SEED)

## 1. Who Gets Into Your Data: Population, Frame, Sample

**Guiding question:** *Which group of people does your evidence actually speak
for: the one you wish you studied, or the one your procedure really reached?*

> *"Every result you will ever cite to me answers a question about some group
> of people. My first question is always the same: which group? Not the group
> you wish you studied. The one your procedure actually reached."*
> — a **policy stakeholder** deciding whether your evidence applies to their city

Your **data strategy** is your plan for how evidence comes to exist: who ends
up in your data, and what happens to them once they are there. In plain terms,
it is the machinery that turns people out in the world into rows in a
spreadsheet. "I will survey 100 dorm residents drawn at random from the housing
list" is a data strategy in one sentence. It has more moving parts than it
first looks. Here is the whole thing on one map. You do not need every piece
today, but keep it in view, because everything from here to your poster is
built on it:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/rdss_fig_8_1.png" width="70%"/></center>

*A data strategy is a chain of linked steps. Sampling (S) draws who is seen.
Treatment assignment (Z) sets what they are exposed to. Response (R) governs
who actually answers. A measurement tool (Q) turns each person's latent outcome
into the observed outcome (Y) you finally analyze, with unobserved
heterogeneity (U) shaping outcomes behind the scenes. The three shaded nodes,
S, Z, and Q, are the handles you actually control; these two lectures build the
first two. (Figure from the replication materials of Blair, Coppock & Humphreys
(2023),* Research Design in the Social Sciences*, an MIT-licensed archive; the
book is free at book.declaredesign.org.)*

Lecture 1 is about **sampling**: the step of your data strategy that decides
which units you get to observe at all. Three nested ideas keep sampling honest.
Picture three boxes, one inside the next:

- **Population**: everyone your question is *about*. Say, all registered
  voters in Los Angeles. The outermost box.
- **Sampling frame**: the actual list you can draw names from, such as a voter
  file. It sits *inside* the population and rarely fills it. Whoever is not on
  the list cannot be sampled, no matter how good your procedure is.
- **Sample**: the units you end up observing. The innermost box.

A claim is honest only when the box it describes matches the box it names. You
will work with a real **frame** today: an extract of the Los Angeles voter
file. For this notebook, treat the file itself as your known world. That way
you can *see* the truth and watch different sampling procedures try to recover
it.

> **A question that often comes up here:** *"My list is huge. Isn't the frame
> basically the population at that point?"* Almost never, and size is exactly
> what hides the gap. A voter file can hold millions of names and still
> systematically miss the people who just moved, just turned eighteen, or were
> purged last month. If those missing people differ on what you are measuring,
> no amount of list-length closes the gap. The frame is the population *minus
> everyone the list forgot*. Your job is to name who that is before you sample
> a single row.

In [ ]:
# Load the frame and confirm its shape before trusting anything.
voters = load_course_data("la_voter_file.csv")
assert voters.shape == (1000, 4), "unexpected shape — flag this!"
print("✓ Loaded la_voter_file.csv —", voters.shape[0], "rows,", voters.shape[1], "columns")
print()
print("Columns:", list(voters.columns))
print()
true_mean_age = voters["age"].mean()
print(f"The whole frame (our 'known world'):")
print(f"  mean age     = {true_mean_age:.2f}   (min {voters['age'].min()}, max {voters['age'].max()})")
print(f"  turnout 2012 = {voters['voted_2012'].mean():.1%} voted")
print(f"  party mix    = {voters['party'].value_counts().head(4).to_dict()}")

### 🔍 Reading the Evidence: name the three boxes

In the cell below, do three things. First: for the LA voter file, say which box
the file itself is (the population, the frame, or a sample) and defend the
label. Second: name one kind of person who is in the *population* "registered
LA voters" but could be missing from the frame. Third: the frame's mean age is
about 48.8. Is that the population's mean age? State the one word that
separates "the frame's average" from "the population's average," and say why
you cannot skip it.

### YOUR ANSWER HERE:

**Which box is the voter file, and why:**

**Someone in the population but missing from the frame:**

**Is 48.8 the population's mean age? The word I cannot skip:**

---

## 2. Random Samples Cluster; Convenience Samples Can Lie

**Guiding question:** *Why can you trust a random sample to speak for a
population you cannot see, while a convenience sample can mislead you no matter
how large it grows?*

You know this frame's true mean age (48.8). Now watch two ways of sampling from
it try to recover that truth. The goal is to see which one you can trust when
you do *not* already know the answer.

**Random sampling** means chance, not convenience, decides who gets into your
data: every unit on the frame has a known chance of being drawn. Example:
numbering all 1,000 rows of the voter file and letting the computer pick 100 at
random. It is not a single move but a small family of procedures, sorted by
*how* you draw and *what unit* you draw:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/rdss_fig_8_2.png" width="72%"/></center>

*The menu of random-sampling procedures: simple, complete, and stratified
drawing (the columns) crossed with sampling individuals, whole clusters, or in
multiple stages (the rows). Each small square is a unit, shaded when it is
drawn. The demo below uses the simplest cell of this grid, simple random
sampling of individuals. (Figure from the replication materials of Blair,
Coppock & Humphreys (2023),* Research Design in the Social Sciences*, an
MIT-licensed archive; the book is free at book.declaredesign.org.)*

First, random sampling in action: draw 100 voters at random, ten separate
times, and watch how much the ten sample means disagree.

### 🔮 Pause & Predict

You are about to draw **10 random samples of 100 voters each** and compute each
sample's mean age. The frame's true mean is 48.8. **Before running the next
cell**, commit to a range: between what low and what high value do you think
the ten sample means will fall? Will they be scattered wildly, or huddled near
48.8?

### YOUR ANSWER HERE:

**My predicted range for the 10 sample means (low to high):**

**Scattered or huddled, and why:**

---

### 🛠️ Run the Study: draw ten random samples

Run the cell below. It draws ten independent random samples of 100 voters from
the frame. It computes each sample's mean age, then plots the ten means against
the frame's true mean. Read the spread before you read the next markdown cell.
If you want, change `n=100` to `n=500` and rerun to feel the spread tighten.

> 💡 **Gemini Prompt:** "Here is a Python cell that draws ten random samples of
> 100 voters and plots each sample's mean age against the true mean: [paste the
> next cell]. Explain what 'sampling variability' means here, and why the ten
> dots land in a band around the true line instead of all on top of it."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check Gemini's explanation against your actual plot. Are the ten means
>   it describes really scattered in a band around the red true-mean line?
> - [ ] Confirm the lowest-to-highest range Gemini claims matches the `lowest`
>   and `highest` values your cell actually printed, not a number it guessed.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# Ten random samples of n=100, each with its own reproducible seed.
sample_means = []
for i in range(10):
    draw = voters["age"].sample(n=100, random_state=SEED + i)
    sample_means.append(draw.mean())

sample_means = np.array(sample_means)
print("Ten random-sample mean ages (n = 100 each):")
print(np.round(sample_means, 2).tolist())
print(f"\n  lowest  = {sample_means.min():.2f}")
print(f"  highest = {sample_means.max():.2f}")
print(f"  true frame mean = {true_mean_age:.2f}")

fig, ax = plt.subplots()
ax.scatter(sample_means, np.zeros_like(sample_means), s=90, color="#4C72B0",
           zorder=3, label="each random sample's mean")
ax.axvline(true_mean_age, color="#C44E52", linestyle="--", linewidth=2,
           label=f"true frame mean = {true_mean_age:.1f}")
ax.set_yticks([])
ax.set_xlim(35, 55)
ax.set_xlabel("Sample mean age")
ax.set_title("Random samples huddle around the truth (n = 100, 10 draws)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Self-check: every random-sample mean lands close to the truth.
assert sample_means.min() > 42 and sample_means.max() < 55, "spread wider than expected"
assert abs(sample_means.mean() - true_mean_age) < 3, "random samples should center on the truth"
print(f"✓ Self-check passed: all 10 random-sample means fall in "
      f"[{sample_means.min():.1f}, {sample_means.max():.1f}] — a tight band around {true_mean_age:.1f}.")

### 🔍 Reading the Evidence: what the spread is telling you

The ten means all landed within a few years of 48.8. They *huddle*. In the cell
below, write two things. First: because random samples cluster this tightly
around the truth, what can a single random sample of 100 honestly claim about
the frame's mean age? Name the missing ingredient that turns "my sample mean"
into "a statement about the frame." Second: if you widened each sample from 100
to 500, would you expect the ten means to spread *more* or *less*? Say why that
matters for how confident you get to sound.

### YOUR ANSWER HERE:

**What one random sample of 100 can honestly claim (and the missing ingredient):**

**Wider samples (n = 500): more or less spread, and why it matters:**

---

Now the other way of sampling. A **convenience sample** is whoever was easiest
to reach: your friends, your followers, the people who happened to walk by. It
is always more tempting because it is easier. It is always more dangerous
because it hides its bias behind that ease.

> **A question that often comes up here:** *"My convenience sample is big.
> Doesn't a big sample fix the bias?"* No, and this is the trap. Size shrinks
> *noise* (sample-to-sample wobble), not *bias* (a systematic tilt in who you
> reach). A huge convenience sample gives you a very precise estimate of the
> wrong number. The next cell shows exactly that.

> 💡 **Gemini Prompt:** "Here is a cell that builds a 'convenience sample' by
> keeping only one party's registrants from a voter file, then compares its
> mean age to the whole file's: [paste the next cell]. Explain how a sample can
> be large and precise and still systematically wrong, and what 'selection on a
> trait correlated with the outcome' means in plain language."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check Gemini's story against your printout. Does the NPP-only mean it
>   discusses match the biased mean your cell actually reported, and the size
>   of the gap to the frame truth?
> - [ ] Make sure Gemini names the *mechanism* (party correlates with age
>   here), not just "the sample is biased." The mechanism is what you must be
>   able to state in your own words.
> - [ ] Log this use in your AI ledger: tool, task, verification.

---

In [ ]:
# A convenience sample is whoever is easiest to reach, NOT a random draw.
# Step 1: the naive grab — the first 100 rows of the file.
first100 = voters["age"].head(100).mean()
print(f"First 100 rows, mean age: {first100:.2f}   (frame truth: {true_mean_age:.2f})")
print("  → Looks fine! But that is LUCK: this file happens to arrive in random")
print("    order, so grabbing the top rows is accidentally a random sample.")
print("    Real convenience samples are not randomly ordered. They cluster on")
print("    whatever made people reachable — and that is where the bias enters.")
print()

# Step 2: a realistic single-channel convenience sample, built on purpose.
# Party correlates strongly with age here. So imagine your survey circulated
# through one no-party-preference (NPP) young-voters network: the only people
# you reached are NPP registrants.
print("Mean age by party (the mechanism that makes convenience biased):")
print(voters.groupby("party")["age"].mean().round(1).sort_values().loc[["NPP", "DEM", "REP"]].to_string())
print()
convenience = voters[voters["party"] == "NPP"]["age"]
print(f"Single-channel convenience sample (all {len(convenience)} NPP registrants):")
print(f"  mean age = {convenience.mean():.2f}   vs frame truth {true_mean_age:.2f} "
      f"→ off by {convenience.mean() - true_mean_age:+.1f} years")

In [ ]:
# Self-check: the convenience sample sits OUTSIDE the entire random-sample band.
assert abs(convenience.mean() - 37.0) < 0.5, "NPP mean drifted — investigate"
assert convenience.mean() < sample_means.min(), \
    "convenience mean should fall below every random-sample mean"
print(f"✓ Self-check passed: the convenience mean ({convenience.mean():.1f}) is below "
      f"the lowest random-sample mean ({sample_means.min():.1f}) — this is bias, not noise.")

### 🔍 Reading the Evidence: what the convenience sample would have you believe

The NPP-only sample would tell you LA voters average about 37, nearly 12 years
younger than the truth. No amount of extra NPP registrants would fix it. A
bigger single-channel sample just nails the wrong number more precisely. In the
cell below, write: (1) the false claim this convenience sample invites, (2) the
*mechanism* that biased it (why did reaching one party skew age?), and (3) one
honest sentence the convenience sample CAN support without lying.

### YOUR ANSWER HERE:

**The false claim it invites:**

**The mechanism behind the bias:**

**One honest sentence it can still support:**

---

### 📝 Practice: name the frame, the sample, the population

For each study below, name the three boxes: the **population** the claim
silently invokes, the **frame** actually drawn from, and the **sample**
observed. Then flag the gap that would most threaten the claim. Answer in the
cell that follows.

- **A.** "We surveyed 300 shoppers outside one Whole Foods and concluded that
  Americans support a soda tax."
- **B.** "From the university's full enrollment roster we randomly emailed
  500 undergraduates; 210 replied. We report undergraduate study habits."
- **C.** "An online poll on a news site's homepage drew 40,000 responses; the
  site reports what 'the public' thinks of the mayor."


### YOUR ANSWER HERE:

**A (population / frame / sample / worst gap):**

**B (population / frame / sample / worst gap):**

**C (population / frame / sample / worst gap):**

---

### ⚖️ Make a Design Choice: random or convenient, and who gets excluded?

For your own project, you will almost never get a perfect random sample of your
ideal population. The honest move is to name the trade you are making. In the
cell below: describe the **most convenient** sample you could realistically
gather, then the **most random** one you could realistically manage, and choose
between them. Defend the choice against one question: *who does my chosen
sample exclude, and does that exclusion correlate with my outcome?*

> 💡 **Gemini Prompt:** "I am studying [your topic] and
> my outcome is [your outcome]. My easiest sample to recruit would be [your
> convenience source]. List the specific groups that recruitment channel would
> systematically under-reach, and for each, explain whether being under-reached
> would bias my estimate of the outcome up or down."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] For each group the AI names, decide *yourself* whether the exclusion
>   plausibly correlates with your outcome. The AI can list groups; only you
>   know your outcome's mechanism.
> - [ ] Cross-check one claimed bias direction against your own reasoning or a
>   real source before you write it into M05.
> - [ ] Log this use in your AI ledger: tool, task, verification.

### YOUR ANSWER HERE:

**My most convenient sample:**

**My most random realistic sample:**

**My choice, and who it excludes (does the exclusion correlate with my outcome?):**

---

### 🎯 Project Transfer: the sampling paragraph of your M05

Write the first paragraph of your M05 Data Strategy now. In the cell below,
state: (1) the **population** your project is about; (2) the **frame** you can
actually reach; (3) the **procedure** you will use to sample from it; and (4)
the **coverage gap**: who is in your population but not your frame, and how you
will disclose it. This paragraph is what lets a reader know exactly whom your
evidence speaks for.

### YOUR ANSWER HERE:

**Population:**

**Frame I can reach:**

**Sampling procedure:**

**Coverage gap (who is missed, and how I'll disclose it):**

---

### 🎟️ Claim Ticket

**Claim Ticket #1** — in one sentence, name the **population** your project
claims to speak about and the **frame** you can actually reach, and state the
honest gap between them.

### YOUR ANSWER HERE:

**My population, my reachable frame, and the gap between them:**

---

---

*(Lecture 2 of 2 starts here.)*

## 3. The Two Randomizations: Who We See vs What They Get

**Guiding question:** *Sampling and assignment both run on chance. What
different job does each one actually do for your claim?*

> *"I keep meeting projects that randomized the wrong thing, or thought one
> randomization did both jobs. Sampling and assignment are different tools for
> different claims. Tell me which one your question needs before you tell me
> you randomized."*
> — a **skeptical peer** who has read too many confident, confused designs

There are **two** randomizations in research, and they answer different
questions:

- **Random sampling** decides **who you SEE**. It draws units from a frame so
  the sample resembles the population. Its payoff is **generalization**:
  results that speak beyond the sample. Example: a pollster dialing 800
  randomly chosen numbers from a state voter file.
- **Random assignment** decides **what they GET**. Among the units you already
  have, it splits them into treatment and control by chance. Its payoff is a
  **fair causal comparison**: the groups are alike *except* for the treatment,
  so a difference afterward can be credited to the treatment. Example: a coin
  flip deciding which half of a workshop gets a new planner app.

You can have either, both, or neither. A survey of a random sample generalizes
but shows no causation. A randomized experiment on a convenience sample
supports a causal claim *within* that sample but may not generalize. Knowing
which randomization your question needs is the whole skill.

> **A question that often comes up here:** *"If I randomly assign my
> participants, haven't I already randomly sampled them too?"* No, and
> collapsing the two is the most common confusion in this whole unit. Random
> *assignment* shuffles the people you already have into treatment and control.
> That earns a fair causal comparison *within that group*; it says nothing
> about whether that group resembles anyone else. Random *sampling* is the
> separate move that earns the right to generalize beyond them. You can
> randomly assign a convenience sample all day and still have zero license to
> speak about the wider population.

### 🔮 Pause & Predict

You will take **100 units** with a known pre-study "motivation" score and split
them into treatment and control **three ways**: (a) an independent coin flip
per unit, (b) *complete* random assignment (exactly 50 and 50, by a shuffled
draw), and (c) no randomization at all (the 50 most-motivated units simply opt
into treatment). The goal of assignment is **balance**: similar average
motivation in both arms *before* any treatment. **Before running**, predict
which of the three will give the most balanced groups, and which the least.

### YOUR ANSWER HERE:

**Most balanced method / least balanced method, and why:**

---

### 🛠️ Hands-On: assign 100 units three ways and check balance

Run the cell below. It splits the same 100 units by all three procedures and
reports the group sizes and the average pre-study motivation in each arm. The
column to watch is the last one: the imbalance the assignment left behind.

> 💡 **Gemini Prompt:** "Here is a cell that assigns 100 units to treatment and
> control three ways (an independent coin flip, complete random assignment, and
> a 'most-motivated units opt in' rule) and reports the group sizes and the
> average baseline motivation in each arm: [paste the next cell]. Explain why
> 'balance' on a pre-treatment variable is the thing that makes a later
> comparison fair."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Read the printed table yourself: confirm which procedure left the
>   smallest `imbalance (T−C)` and which left the largest, and check Gemini
>   agrees with the numbers you see rather than the other way around.
> - [ ] Check Gemini's account against the table. Does the coin flip drift in
>   group *size* while the opt-in rule drifts in *motivation*, exactly as your
>   output shows?
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# Three assignment procedures on the same 100 units. Re-seed for a
# self-contained, reproducible demo (all randomness flows from SEED).
rng = np.random.default_rng(SEED)
N = 100
baseline = rng.normal(50, 10, N).round(1)          # pre-study motivation score

coin = rng.binomial(1, 0.5, N)                      # (a) independent coin flip
perm = rng.permutation(N)                           # (b) complete: shuffle, treat first 50
complete = np.zeros(N, dtype=int); complete[perm[:50]] = 1
none = (baseline >= np.median(baseline)).astype(int)  # (c) none: most-motivated opt in

def balance(z):
    z = np.asarray(z)
    return pd.Series({
        "n_treated": int(z.sum()),
        "n_control": int((z == 0).sum()),
        "mean_motivation_T": round(baseline[z == 1].mean(), 2),
        "mean_motivation_C": round(baseline[z == 0].mean(), 2),
        "imbalance (T−C)": round(baseline[z == 1].mean() - baseline[z == 0].mean(), 2),
    })

table = pd.DataFrame({
    "coin-flip": balance(coin),
    "complete": balance(complete),
    "none (opt-in)": balance(none),
}).T
print("Covariate balance under three assignment procedures (100 units):")
print(table.to_string())

In [ ]:
# Self-check: complete assignment balances best; opt-in shatters balance.
assert table.loc["complete", "n_treated"] == 50, "complete must be exactly 50/50"
assert abs(table.loc["complete", "imbalance (T−C)"]) < abs(table.loc["none (opt-in)", "imbalance (T−C)"])
assert table.loc["none (opt-in)", "imbalance (T−C)"] > 10, "opt-in should be badly imbalanced"
print("✓ Self-check passed: complete random assignment is nearly balanced "
      f"({table.loc['complete', 'imbalance (T−C)']:+.2f}); the coin flip drifts in group SIZE "
      f"({table.loc['coin-flip','n_treated']:.0f}/{table.loc['coin-flip','n_control']:.0f}); "
      f"opt-in is wrecked ({table.loc['none (opt-in)','imbalance (T−C)']:+.2f}).")

### 🔍 Reading the Evidence: what each procedure buys

Read the table. **Complete** assignment gave exactly 50/50 groups with almost
identical average motivation. The **coin flip** balanced motivation reasonably
but drifted in group *size*, because it does not guarantee 50/50. **No
randomization** kept the sizes at 50/50 but left the treatment group far more
motivated to begin with. In the cell below, write: (1) why the opt-in group's
head start on motivation would make a later "the treatment worked!" claim
untrustworthy, and (2) which randomization this whole exercise is about, the
who-you-SEE one or the what-they-GET one, and how you know.

### YOUR ANSWER HERE:

**Why opt-in wrecks a later causal claim:**

**Which randomization is this (see vs get), and how I know:**

---

### 📝 Practice: the two-randomizations quiz

For each design, say which randomization (if either) is present, choosing
**sampling**, **assignment**, **both**, or **neither**, and say what it buys.
Answer in the cell that follows.

- **A.** A polling firm randomly dials phone numbers from a full state voter file
  and asks who people will vote for.
- **B.** A teacher takes the 60 volunteers who signed up for a study-skills workshop
  and, by a coin flip, gives half a new planner app; grades are compared after.
- **C.** Researchers randomly sample 800 households nationwide, then within each
  household randomly assign one adult to receive a vaccine-reminder text.
- **D.** A blogger surveys their own newsletter subscribers about media habits.


### YOUR ANSWER HERE:

**A:** &nbsp; **B:** &nbsp; **C:** &nbsp; **D:**

**What each present randomization buys:**

---

## 4. Feasibility and Ethics: Can You Do It, and Should You?

A design can be sound and still impossible. It can also be possible and still
wrong to run. Two checks close out your M05: a feasibility triage and an ethics
checkpoint. Both are text-first, honest, and practical.

**Feasibility** is the question of whether your study can actually be completed
with the time, skills, money, and access you really have. Example: a survey of
40 dorm residents fits a semester; a three-year national panel does not. Triage
your design with four questions, answered plainly:

1. **Data in time?** Can the data you need actually *exist*, collected,
   cleaned, and in hand, with weeks to spare before the end of the semester?
   A design whose data cannot exist in time is not a design.
2. **Skills?** Can you, with the tools in this course and AI help you verify,
   do what the analysis requires? Or is there a lighter version you *can* do
   well?
3. **Cost?** Time, money, access, participants: what does it take, and do you
   have it?
4. **Consent?** If people are involved, can you obtain informed consent and
   handle their data responsibly?

A single "no" is not a death sentence. It is a redesign prompt. The most common
honors-scale fix is to **shrink the scope** or **switch to secondary data**.

### The Ethics Checkpoint: a decision tree for research with people

> *"'It's just a class project' is the sentence I hear right before the
> projects that need the most review. The scale of a study does not set its
> ethical footprint. Its contact with human beings does. Walk the tree
> honestly, and most of you will land in the safe zone. But you have to walk
> it."*
> — an **IRB reviewer**, on the reflex to skip the question

**Ethics review** is a check, done *before* you collect data, that your study
treats the people in it fairly and safely. At Purdue it runs through the
**IRB** (Institutional Review Board), the university committee that reviews
research involving human beings; Purdue's office for this is the Human Research
Protection Program (HRPP). Example: a short survey about study habits gets a
light look, while an experiment that changes what people eat gets serious
scrutiny.

Walk your project down this tree:

- **Does your project use only secondary data**, meaning data already
  collected, public or properly de-identified? Think a government dataset, or
  a published replication file like the ones in this course. → This is the
  **honors-scale default**, and it is typically **not** human-subjects
  research requiring review. You still owe honest provenance and respect for
  the original participants.
- **Does it collect new data from living people** via surveys or interviews?
  → Likely **minimal risk** (no more than everyday life). But this is where
  you check in with Purdue's HRPP (IRB) *before* collecting, plan **informed
  consent**, and protect identifiers.
- **Does it intervene**, changing something people experience to see the
  effect? → Highest scrutiny. Do not run it until you have IRB guidance. The
  breakfast-vs-exam idea a classmate floats below is exactly this branch.

**Minimal risk** means the probability and magnitude of harm are no greater
than in daily life. It is a real category with real obligations, not a
loophole.

There is a second axis the tree does not show: how much your inquiry is
*worth*, weighed against what it costs the people in it. The book draws exactly
that trade-off:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/rdss_fig_21_1.png" width="68%"/></center>

*Two things climb as a study grows: the ethical cost of involving more people
(the straight line) and the scientific benefit of a larger sample (the curved
lines, one per level of inquiry importance). A study is justified only in the
shaded region, where its benefit curve sits above the cost line, and only a
sufficiently important inquiry (the upper curve) clears it comfortably. The
bigger and more important the inquiry, the more design care it takes for the
benefits to genuinely exceed the ethical costs. (Figure from the replication
materials of Blair, Coppock & Humphreys (2023),* Research Design in the Social
Sciences*, an MIT-licensed archive; the book is free at book.declaredesign.org.)*

> **A question that often comes up here:** *"My project only touches public,
> secondary data. Do I really have any ethics obligation at all?"* Yes, a real
> if lighter one. Using secondary data usually keeps you off the branches that
> need IRB review. But the people in that dataset were real, were promised
> something when it was collected, and cannot re-consent to your reuse. So you
> still owe honest provenance, respect for the original consent, and care not
> to re-identify anyone a dataset worked to anonymize. "Public" lowers the
> ethical cost; it does not zero it out.

### ⚖️ Make a Design Choice: walk a real project down the ethics tree

A classmate wants to test whether skipping breakfast hurts exam scores, by
asking friends to skip breakfast before a real exam. In the cell below: (1)
place this on the ethics tree (secondary / survey / intervention); (2) name the
specific reason it needs more than a shrug; (3) propose the **redesign** that
answers a related question at the honors-scale default (secondary data or a
minimal-risk survey), and say what the redesign gives up and what it keeps.

### YOUR ANSWER HERE:

**Where it sits on the tree:**

**Why it needs real review:**

**My honors-scale redesign (what it gives up / keeps):**

---

### 🎯 Project Transfer: the feasibility and ethics paragraphs of your M05

This is where M05 comes together. In the cell below, finish your Data Strategy
by writing two short paragraphs for *your* project. **Feasibility:** answer all
four triage questions (data in time, skills, cost, consent) and name your
single biggest risk. **Ethics:** state where your project sits on the tree, its
risk classification, and the one safeguard or redesign you are committing to.
Combined with your sampling paragraph from Lecture 1 and your assignment
reasoning today, these paragraphs *are* your M05 draft. The async module at the
end of the week is where you stitch them together and record your 90-second
statement for the discussion board.

### YOUR ANSWER HERE:

**Feasibility (data in time / skills / cost / consent + biggest risk):**

**Ethics (tree location / risk classification / safeguard or redesign):**

---

### 🎟️ Claim Ticket

**Claim Ticket #2** — state your project's **ethics classification** (secondary /
minimal-risk survey / intervention) and the **one feasibility risk** that most
worries you, in one sentence each.

### YOUR ANSWER HERE:

**My ethics classification:**

**The feasibility risk that most worries me:**

---

## 5. Wrap-Up

Across two lectures you built MIDA's Data strategy from the ground up. You
separated the three boxes: population, frame, sample. You watched random
samples huddle around the truth while a convenience sample missed it by nearly
twelve years, biased not by bad luck but by *who it could reach*. You pulled
apart the two randomizations, sampling for who you see and assignment for what
they get, and saw complete random assignment balance groups that opt-in
wrecked. And you closed with the two questions no design escapes: can you
actually do it, and should you?

> **"Your data strategy is a promise about who your evidence speaks for and
> what you owed the people in it. Break either half quietly, and every result
> you report is borrowing against a debt you never disclosed."**

Next comes the week's async module, nb08. It hands you a real published claim
to take apart, kindly and with evidence, using everything from the inquiry
compass to the sampling and assignment logic you just built. It also includes
the block that stitches your sampling, assignment, feasibility, and ethics
paragraphs into your finished M05: a 90-second recorded statement you post to
the discussion board. Bring your draft paragraphs; the assembly goes fast when
the pieces are ready.

---

## 6. Sources & Provenance

**Provenance of this notebook:**
- *RDSS ch. 8 'Crafting a data strategy' §8.1 (sampling) + assignment sections | sampling & assignment within a data strategy | frame/sample/population framing, simple-random-sampling and complete-random-assignment simulations | translated (R→Python) & adapted for the honors non-quant audience*
- *declaration_2.1 (concept) | RDSS ch. 8 | random sampling declared as a data-strategy step | translated (concept-level, no code reproduced)*
- *la_voter_file.csv | rdss package data | party/age/tract/turnout used as a known frame for the sampling demonstrations | adapted (the file stands in as the 'known world')*
- *fresh | two-randomizations framing + honors-scale feasibility triage + ethics decision tree | n/a | newly-constructed-from-source-concept*

**Dataset attribution:** Dataset from the `rdss` package (Blair, Coppock &
Humphreys, MIT License), companion to *Research Design in the Social Sciences*
(2023).

**Readings:**
- Blair, G., Coppock, A., & Humphreys, M. (2023). *Research Design in the
  Social Sciences*, ch. 8 'Crafting a data strategy' (§8.1 sampling; assignment
  sections). Free online: [book.declaredesign.org](https://book.declaredesign.org/).
- Purdue University Human Research Protection Program (IRB): guidance for
  course-based and undergraduate research (consult the current Purdue HRPP
  pages directly before collecting any new data from people).

---

<center>

Thank you!

</center>